# Corbel YOLOv8 Run B: Synthetic Data Fine-Tuning

Build on Run A (yolov8s transfer learning, 81.0% overall F1) by adding 800 synthetic procedurally-generated floor plans.

**Strategy**: Merge synthetic images into training set, continue warm-start fine-tuning (100 epochs at lr0=0.001).

**Expected**: Door/Window should improve toward 90%+ (wall limited to ~80-88% by geometry).

## Cell 1: GPU Check & Mount Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Dataset Setup

In [ ]:
!unzip -q /content/drive/MyDrive/fyp-dataset/corbel-dataset.zip -d /content/
!sed -i 's|^path:.*|path: /content/dataset|' /content/dataset/cubicasa.yaml
print("✓ Base dataset ready at /content/dataset")
print(f"  images/train: {len(__import__('os').listdir('/content/dataset/images/train'))} files")
print(f"  labels/train: {len(__import__('os').listdir('/content/dataset/labels/train'))} files")

## Cell 3: Install Ultralytics

In [ ]:
!pip install -q ultralytics
print("✓ Ultralytics installed")

## Cell 4: Upload & Merge Synthetic Images

**First**: Upload `ml/synth/synth_data.zip` (1.1 MB) via Colab's file sidebar (Files icon > Upload) to `/content/synth_data.zip`, then run the cells below.

In [ ]:
import zipfile
import os

zip_path = '/content/synth_data.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/synth')
    print(f"✓ Extracted synth data")
    print(f"  images: {len(os.listdir('/content/synth/images'))}")
    print(f"  labels: {len(os.listdir('/content/synth/labels'))}")
else:
    print(f"⚠ {zip_path} not found. Upload it via the Files sidebar first.")

In [ ]:
import os
import shutil

synth_images = '/content/synth/images'
synth_labels = '/content/synth/labels'
train_images = '/content/dataset/images/train'
train_labels = '/content/dataset/labels/train'

if os.path.exists(synth_images):
    print(f"Merging {len(os.listdir(synth_images))} synthetic images...")
    for f in os.listdir(synth_images):
        shutil.copy(os.path.join(synth_images, f), os.path.join(train_images, f))
    for f in os.listdir(synth_labels):
        shutil.copy(os.path.join(synth_labels, f), os.path.join(train_labels, f))
    print(f"✓ Merged. Train set now has {len(os.listdir(train_images))} images")
else:
    print("⚠ Synthetic images not found at /content/synth/")
    print("  (This is expected if synth files will be uploaded separately)")

## Cell 5: Fine-Tune with Synthetic Data (100 epochs)

In [ ]:
from ultralytics import YOLO
import os

# Load Run A checkpoint
runA_path = '/content/drive/MyDrive/corbel-runs/corbel-runA-transfer/weights/best.pt'

if os.path.exists(runA_path):
    print(f"Loading Run A checkpoint from {runA_path}...")
    model = YOLO(runA_path)

    print("\nFine-tuning yolov8s on CubiCasa5k + 800 synthetic images (100 epochs, lr0=0.001)...\n")
    model.train(
        data='/content/dataset/cubicasa.yaml',
        epochs=100,           # Warm start from Run A
        patience=40,
        imgsz=640,
        save_period=1,
        optimizer='AdamW',
        batch=-1,
        lr0=0.001,            # Warm start
        lrf=0.0001,
        warmup_epochs=5,
        # Augmentation (tuned in Run A)
        copy_paste=0.3,
        mixup=0.1,
        degrees=5.0,
        erasing=0.25,
        project='/content/drive/MyDrive/corbel-runs',
        name='corbel-runB',
        exist_ok=True,
    )
    print("\n✓ Run B training complete!")
else:
    print(f"ERROR: Run A checkpoint not found at {runA_path}")
    print("Make sure Run A completed and its weights were saved to Drive.")

## Cell 6: Export ONNX

In [ ]:
from ultralytics import YOLO
import os

best_path = '/content/drive/MyDrive/corbel-runs/corbel-runB/weights/best.pt'

if os.path.exists(best_path):
    print(f"Exporting ONNX from {best_path}...\n")
    model = YOLO(best_path)
    onnx_path = model.export(format='onnx', imgsz=640, opset=17)
    print(f"\n✓ ONNX exported: {onnx_path}")
    print(f"\nNext steps:")
    print(f"1. Download the ONNX from Drive to local: ml/train4.onnx")
    print(f"2. Benchmark: node --experimental-strip-types corbel/scripts/benchmark-local-yolo.ts --split test --n 400 --model ml/train4.onnx")
    print(f"3. Compare against Run A (81.0% overall F1)")
else:
    print(f"ERROR: best.pt not found at {best_path}")